# CampusAid policy Q&A — evaluator fallback

This public Colab notebook is a lightweight fallback because the Render free service exceeds its 512 MiB memory limit at startup. It exercises the **actual CampusAid policy retrieval and answer module**, not a hardcoded mock. It covers policy Q&A only; it does not provide the full grievance submission/tracking web interface.

- Open and inspect the saved live run below without an API key.
- To try a new question, run the setup cell, then the interactive query cell. It will ask for a Gemini API key using a hidden prompt; do not paste a key into notebook code or save it in the notebook.
- Notebook link: [Open in Google Colab](https://colab.research.google.com/github/akritah/CampusAid/blob/codex/campusaid-evaluator/code/CampusAid_Evaluator_Demo.ipynb).
- The evaluation prompt and all eight actual responses are saved in `test-data/policy_qa_run_2026-11-24.json` in the linked public repository. Current fee-deadline/conflict cases are marked not testable because the required notices are not in the indexed corpus.

The attendance source is based on Section 10 of Bennett University's [2024–25 Examination Manual](https://www.bennett.edu.in/wp-content/uploads/2025/01/Examination-Manual-2024-25-.pdf). Confirm current-year policies with the university.


In [ ]:
# Install only the light dependencies used by the policy QA module (not Torch/Whisper).
%pip -q install fastapi pypdf requests

from pathlib import Path
import os, subprocess, sys, getpass, json

REPO = Path('/content/CampusAid')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'codex/campusaid-evaluator',
        'https://github.com/akritah/CampusAid.git', str(REPO)
    ], check=True)
sys.path.insert(0, str(REPO / 'backend'))
os.environ['POLICY_STORAGE_DIR'] = str(REPO / 'backend' / 'policy_documents')
from app import policy_qa
policy_qa.POLICY_DIR = Path(os.environ['POLICY_STORAGE_DIR'])


def ask_campusaid(question):
    # Input is hidden and held only in this Colab runtime; it is not written to disk.
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key (hidden): ')
    os.environ.setdefault('GEMINI_MODEL', 'gemini-3.5-flash-lite')
    matches = policy_qa.retrieve(question)
    chunks = []
    for i, (chunk, score) in enumerate(matches, 1):
        first_line = chunk.text.splitlines()[0] if chunk.text.splitlines() else ''
        chunks.append({
            'chunk_id': f'C{i}', 'document': chunk.source,
            'section_or_heading': first_line[:160],
            'relevance': round(score, 3), 'text': chunk.text
        })
    response = policy_qa.answer(question)
    return {'input': question, 'retrieved_chunks': chunks, 'actual_output_json': response}


## Saved actual run

This is a preserved response from an actual local run on 2026-11-24. It does not call Gemini when the cell is opened or re-run; the JSON run artifact in the repository is the source of these outputs. Review failed and not-testable cases as recorded.


In [ ]:
from IPython.display import display, Markdown
run_path = REPO / 'test-data' / 'policy_qa_run_2026-11-24.json'
SAVED_SAMPLE = json.loads(run_path.read_text(encoding='utf-8'))['results'][0]
SAVED_SAMPLE = {'input': SAVED_SAMPLE['input'], 'retrieved_chunks': SAVED_SAMPLE['retrieved_chunks'], 'actual_output_json': SAVED_SAMPLE['actual_output_json']}
saved_run = json.loads(run_path.read_text(encoding='utf-8'))
rows = ['| Test | Result | Retrieved documents | Actual answer (short) |', '|---|---|---|---|']
for item in saved_run['results']:
    docs = ', '.join(c['document'] for c in item['retrieved_chunks']) or 'None'
    short = item['actual_output_json'].get('answer', 'HTTP ' + str(item['actual_output_json'].get('http_status'))).replace('|', r'\|')
    rows.append(f"| {item['id']} | {item['observed_status']} | {docs} | {short} |")
display(Markdown('\n'.join(rows)))
print('Sample run — exact recorded input, retrieved chunks and CampusAid JSON:')
print(json.dumps(SAVED_SAMPLE, ensure_ascii=False, indent=2))


CampusAid policy Q&A — recorded live evaluation (2026-11-24)
Run: backend/app/policy_qa.py + indexed repository documents + configured Gemini API. Key excluded.
Summary: 1 pass, 5 fail, 2 not testable.

T1 [PASS] What is the minimum attendance required to sit for end-semester exams?
Retrieved: Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt
Actual: The minimum attendance required to sit for end-semester examinations is 75% in each course during a semester [Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt].

T2 [FAIL] How do I apply for re-evaluation of my exam paper?
Retrieved: Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt
Actual: The documents do not specify how to apply for the re-evaluation of an exam paper. Please contact the relevant examination office for this information.

T3 [FAIL] What is the procedure for taking leave from the hostel?
Retrieved: no chunk
Actual: To take leave from a university hostel, students typically need to submit a fo

## Try a new question

Run this cell after setup to query the same CampusAid retriever and answer function with a question of your choice. A Gemini API key is required for new generated answers; it is prompted privately. No key is stored in the notebook.


In [ ]:
question = input('Ask a university policy question: ')
result = ask_campusaid(question)
print(json.dumps(result, ensure_ascii=False, indent=2))
